# Fine-tuning ByT5 pour normaliser le wolof

Ce notebook entraîne `google/byt5-small` à passer d'un wolof écrit « à la française / SMS » à l'orthographe officielle.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Exécution → Tout exécuter*.

Durée indicative : ~1 h sur T4, ~20 min sur L4/A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code et dépendances

In [ ]:
!git clone -q https://github.com/LahadMbacke/wolof-normalizer.git
%cd wolof-normalizer
!pip install -q -e ".[eval]" "transformers==5.20.2" accelerate

## 2. (Optionnel) Sauvegarder sur Google Drive

Colab peut couper la session : en enregistrant les checkpoints sur Drive, on ne perd pas l'entraînement. Mettre `USE_DRIVE = False` pour s'en passer.

In [ ]:
USE_DRIVE = True
OUTPUT_DIR = "runs/byt5-small"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/wolof-normalizer/byt5-small"
print(OUTPUT_DIR)

## 3. Données augmentées

Vraies paires (soynade) + phrases standard (galsenai, Wikipédia wolof) bruitées automatiquement. Les phrases du test sont exclues.

In [ ]:
!python scripts/make_synthetic.py

## 4. (Optionnel) Connexion Hugging Face pour publier le modèle

Renseigner `HUB_MODEL_ID` (ex. `Lahad/wolof-normalizer-byt5-small`) et un token *write*. Laisser `None` pour ne rien publier.

In [ ]:
HUB_MODEL_ID = None  # "Lahad/wolof-normalizer-byt5-small"
if HUB_MODEL_ID:
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Entraînement

In [ ]:
push = f"--push-to-hub {HUB_MODEL_ID}" if HUB_MODEL_ID else ""
!python scripts/train_byt5.py --output-dir "{OUTPUT_DIR}" --epochs 5 --batch-size 16 --grad-accum 2 {push}

## 6. Évaluation sur le test (688 vraies paires jamais vues)

In [ ]:
!python scripts/run_evaluation.py --model "{OUTPUT_DIR}/best"

## 7. Essayer

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(f"{OUTPUT_DIR}/best")
model = AutoModelForSeq2SeqLM.from_pretrained(f"{OUTPUT_DIR}/best").to("cuda").eval()

def normalize(sentences):
    batch = tokenizer(sentences, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**batch, max_length=512, num_beams=4)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

for s, n in zip(examples := [
    "Nanga def ? Mangui fi rek, dieuredieuf",
    "dafa beug wakh ak yaw thi kanam",
    "gnou bari dagnou meuna dem keur gui souba",
], normalize(examples)):
    print(s, "->", n)

## 8. Télécharger le modèle (si pas sur Drive ni Hugging Face)

In [ ]:
!cd "{OUTPUT_DIR}" && zip -qr /content/byt5-wolof-best.zip best
from google.colab import files
files.download("/content/byt5-wolof-best.zip")